# Publish the motor housing to the API server

Kernel: **SysML** -- check the top right, not Python.

One cell defines the model, the next pushes it to the server via `%publish`.
The target comes from `ISYSML_API_BASE_PATH` and must read `http://sysml-api:9000`.

In [1]:
package Test { part def Housing; }

Package Test (5354dbfe-b596-4aac-87b1-7e74ba670fea)


In [2]:
package Test2 { import ISQ::*; }

ERROR:mismatched input 'import' expecting '}' (2.sysml line : 1 column : 17)
ERROR:no viable alternative at input '::' (2.sysml line : 1 column : 27)


In [3]:
package Test3 { private import ISQ::*; }

Package Test3 (e7687465-c320-47c0-967f-6a846bd9ddf3)


In [4]:
%help

The following SysML v2 magic commands are available.
For help on a specific command, use "%help <COMMAND>" or "%<cmd> -h".

%eval		Evaluate a given expression.
%export		Save a file of the JSON representation of the abstract syntax tree rooted in the named element.
%help		Get a list of available commands or help on a specific command
%list		List loaded library packages or the results of a given query
%load		Load models from the repository
%projects	List projects in the repository
%publish	Publish to the repository the model elements rooted in a named element
%repo		Set the API base path for the repository
%show		Print the abstract syntax tree rooted in a named element
%view		Render the view specified by the named view usage
%viz		Visualize the named model elements


In [5]:
package Test {
    part def Housing;
}

Package Test (94ec909d-d4fd-41eb-a9b6-99ac52d64bc8)


In [6]:
package MotorHousing {
    private import ScalarValues::*;
    private import ISQ::*;
    private import SI::*;

    part def Housing {
        attribute length : LengthValue;
        attribute mass   : MassValue;
    }

    requirement def MaxLength {
        doc /* The motor housing must not exceed 120 mm in length. */
        subject housing : Housing;
        attribute maxLength : LengthValue = 120 [mm];
        require constraint { housing.length <= maxLength }
    }

    requirement def MaxMass {
        doc /* The mass of the motor housing must not exceed 0.9 kg. */
        subject housing : Housing;
        attribute maxMass : MassValue = 0.9 [kg];
        require constraint { housing.mass <= maxMass }
    }

    part motorHousing : Housing;

    requirement maxLengthCheck : MaxLength;
    requirement maxMassCheck   : MaxMass;

    satisfy maxLengthCheck by motorHousing;
    satisfy maxMassCheck   by motorHousing;
}

Package MotorHousing (7cbcc1ed-a357-4782-b0bb-44d7b91d74ff)


In [7]:
%publish MotorHousing

API base path: http://sysml-api:9000
Processing.
Posting commit. Additions: 164, Deletions: 0, Modified elements: 0
Successfully posted commit: c42e0ff7-bf2e-4ff8-8818-f3f1955ed911 on branch: (main) cf3a703c-b7f3-4128-b532-960af79d8c46



Saved to Project MotorHousing (fd6a22e8-1b3d-43f8-91e3-98fc5126aab3)


If `%publish` returns a project id, the model is on the server.
Then fetch the elements from the devcontainer and look at the real payload --
that is what `extract._to_requirement` is written against:

```bash
python -c "
from sysmlcad.sysml_client import SysMLClient
c = SysMLClient(); p = c.projects(); print(p)
pid = p[0]['@id']; cid = c.head_commit(pid)
els = c.elements(pid, cid)
print(len(els), 'elements')
print([e['@type'] for e in els][:40])
c.close()"
```